<a href="https://colab.research.google.com/github/LuisFelipeVelasco/ML_And_Generative_AI_Experiments/blob/main/Notebooks/binary_decision_tree_from_scratch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#**BINARY DECISION TREE ALGORITHM FROM SCRATCH**

**IMPORT LIBRARIES**

In [1]:
import numpy as np
import pandas as pd
from google.colab import sheets


**DEFINE CLASS**

In [54]:
import numpy as np
import pandas as pd

class BinaryDecisionTree:
    def __init__(self):
        """
        Initializes the BinaryDecisionTree instance.
        """
        self.tree = {}  # Changed from class variable to instance variable to store tree structure

    def get_valid_labels(self, attribute, labels, rule):
        """
        Filters labels based on a rule applied to an attribute.

        Parameters:
        attribute (numpy.ndarray): The array of attribute values.
        labels (numpy.ndarray): The array of labels corresponding to the attribute values.
        rule (float): The threshold value for filtering the attribute.

        Returns:
        numpy.ndarray: The labels for which the attribute values are less than or equal to the rule.
        """
        mask_valid_labels = attribute <= rule
        return labels[mask_valid_labels]

    def get_gini_impurity(self, labels):
        """
        Calculates the Gini impurity for a given set of labels.

        Parameters:
        labels (numpy.ndarray): The array of labels.

        Returns:
        float: The calculated Gini impurity.
        """
        if len(labels) == 0:
            return 0
        counts_unique_labels = np.unique(labels, return_counts=True)[1]
        total_labels = sum(counts_unique_labels)
        gini_impurity = sum((counts_unique_labels / total_labels) * (1 - (counts_unique_labels / total_labels)))
        return gini_impurity

    def get_metrics_attribute(self, attribute, labels):
        """
        Determines the best splitting rule (attribute value) that minimizes Gini impurity.

        Parameters:
        attribute (numpy.ndarray): The array of attribute values to find the rule for.
        labels (numpy.ndarray): The array of labels corresponding to the attribute values.

        Returns:
        tuple: (rule_attribute, gini_impurity, matching_samples)
               The optimal threshold, its corresponding impurity, and the number of passing samples.
        """
        sorted_indices = np.argsort(attribute)[::-1]
        attribute_sorted = attribute[sorted_indices]
        labels_sorted = labels[sorted_indices]
        old_average = 0
        rule_attribute = 0
        gini_impurity = 1

        for i in range(len(attribute_sorted) - 1):
            average = (attribute_sorted[i] + attribute_sorted[i + 1]) / 2
            if average != old_average:
                old_average = average
                labels_satisfy_attribute = self.get_valid_labels(attribute_sorted, labels_sorted, average)
                new_gini_impurity = self.get_gini_impurity(labels_satisfy_attribute)
                if new_gini_impurity < gini_impurity:
                    gini_impurity = new_gini_impurity
                    rule_attribute = average

        matching_samples = len(labels[attribute <= rule_attribute])
        return rule_attribute, gini_impurity, matching_samples

    def get_properties_best_attribute(self, attributes, labels):
        """
        Iterates through all features to find the best feature and split rule based on Gini impurity.

        Parameters:
        attributes (pandas.DataFrame): The matrix of features/attributes.
        labels (numpy.ndarray): The target labels.

        Returns:
        tuple: (best_attribute_index, rule_best_attribute)
               The column index of the best feature and its optimal split threshold.
        """
        best_attribute = 0
        rule_best_attribute = 0
        gini_impurity_best_attribute = 1
        great_matching_samples = 0

        for i in range(0, len(attributes.columns)):
            attribute = attributes.iloc[:, i].values
            rule_attribute, gini_impurity_attribute, matching_samples = self.get_metrics_attribute(attribute, labels)
            if (gini_impurity_attribute <= gini_impurity_best_attribute and matching_samples > great_matching_samples):
                gini_impurity_best_attribute = gini_impurity_attribute
                rule_best_attribute = rule_attribute
                great_matching_samples = matching_samples
                best_attribute = i

        return best_attribute, rule_best_attribute

**CHARGE DATASET**

In [4]:
!wget https://raw.githubusercontent.com/LuisFelipeVelasco/ML_And_Generative_AI_Experiments/main/Sources/iris.csv
df=pd.read_csv('iris.csv')
sheet = sheets.InteractiveSheet(df=df)

--2026-06-23 14:54:13--  https://raw.githubusercontent.com/LuisFelipeVelasco/ML_And_Generative_AI_Experiments/main/Sources/iris.csv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.111.133, 185.199.110.133, 185.199.108.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.111.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 3858 (3.8K) [text/plain]
Saving to: ‘iris.csv.2’

iris.csv.2          100%[===================>]   3.77K  --.-KB/s    in 0s      

2026-06-23 14:54:13 (48.1 MB/s) - ‘iris.csv.2’ saved [3858/3858]

https://docs.google.com/spreadsheets/d/1Hxy4nFhWNPgVkHJQoLV4QiV4IhggDOrzO7CCKodNeqo/edit#gid=0


**GET ATTRIBUTE RULE**

In [55]:
labels=df["species"].values
attributes=df.iloc[:,:-1]
model=BinaryDecisionTree()
best_attribute , rule=model.get_properties_best_attribute(attributes,labels)
valid_labels=labels[df.iloc[:,best_attribute]<=rule]
unique,count=np.unique(valid_labels,return_counts=True)
print(best_attribute)
print(rule)
print(unique)
print(count)

2
2.45
['setosa']
[50]


In [56]:
print(df.iloc[:,best_attribute])

0      1.4
1      1.4
2      1.3
3      1.5
4      1.4
      ... 
145    5.2
146    5.0
147    5.2
148    5.4
149    5.1
Name: petal_length, Length: 150, dtype: float64
